# Azure AI Foundry - Agent Demo

This notebook shows how to create and query an AI agent with the **Azure AI Projects SDK** and send telemetry to Application Insights and Log Analytics (`AIAgentsInfo`).

**Run order:** execute sections from top to bottom.

## 0. Create/Re-use Virtual Environment & Register Kernel

Create (or re-use) a Python virtual environment to isolate dependencies for this project, then install `ipykernel` and register it as a notebook kernel.

In [ ]:
import subprocess, sys, os

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (no-op if it already exists)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Install / upgrade ipykernel inside the venv
pip = os.path.join(venv_dir, "bin", "pip")
subprocess.check_call([pip, "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
python = os.path.join(venv_dir, "bin", "python")
subprocess.check_call([python, "-m", "ipykernel", "install", "--user", "--name", "ai-agent-demo", "--display-name", "AI Agent Demo (.venv)"])

print(f"✅ Virtual environment created or reused at {venv_dir}")
print("👉 Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

## 1. Install Dependencies

Run this section to install required packages:
- **azure-ai-projects** — Azure AI Foundry projects and agents SDK
- **azure-identity** — `DefaultAzureCredential` authentication
- **azure-monitor-opentelemetry** — Azure Monitor OpenTelemetry distribution
- **opentelemetry-sdk** — Core OpenTelemetry tracing SDK
- **azure-core-tracing-opentelemetry** — Azure SDK tracing bridge

In [ ]:
%pip install --no-input --pre "azure-ai-projects>=2.0.0b4" azure-identity "opentelemetry-sdk<1.39" "opentelemetry-api<1.39" azure-monitor-opentelemetry azure-core-tracing-opentelemetry

## 2. Import Libraries

Run this section after dependencies are installed.

It imports:
- `DefaultAzureCredential` for Azure authentication
- `AIProjectClient` for project operations
- `PromptAgentDefinition` for agent definition
- `AIProjectInstrumentor` (used in Section 3.1 for tracing)

In [ ]:
import os, sys, platform
from azure.core.settings import settings

# Force Azure SDKs to emit OpenTelemetry spans
settings.tracing_implementation = "opentelemetry"

# Set resource detector override BEFORE importing azure-monitor-opentelemetry
# to ensure it takes effect during module initialization
if platform.system() == "Darwin":
    os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
    os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"
    print(f"⚙️  macOS detected — disabled Azure IMDS resource detector to avoid local hang")

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.telemetry import AIProjectInstrumentor

print("✅ All libraries imported successfully")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

## 3. Configure the Project Client

Set your Azure AI Foundry project endpoint in this step, then create an authenticated `AIProjectClient` using `DefaultAzureCredential`.

In [ ]:
import logging
import subprocess

user_endpoint = "https://zolab1702-ai-foundry.services.ai.azure.com/api/projects/zolab1702-ai-fndry-proj"

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=user_endpoint,
    credential=credential,
)

print("✅ AIProjectClient configured")
print(f"   Endpoint: {user_endpoint}")
print("   Auth:     DefaultAzureCredential")

def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None

def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        return _run_command(["az", "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None

# Optional credential-source probe for transparency
try:
    identity_logger = logging.getLogger("azure.identity")
    if not identity_logger.handlers:
        identity_logger.addHandler(logging.StreamHandler())
    identity_logger.setLevel(logging.INFO)

    _ = credential.get_token("https://management.azure.com/.default")
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        print(f"   🔐 Credential used: {credential_name}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            print(f"   👤 Signed-in account: {identity_hint}")
        else:
            print("   👤 Signed-in account: not available for this credential type")
    else:
        print("   🔍 Credential used: check azure.identity INFO logs above")
except Exception as ex:
    print(f"   ⚠️ Credential probe skipped: {type(ex).__name__}: {ex}")

## 3.1 Enable Telemetry

This section wires end-to-end tracing for agent operations and exports spans to Azure Monitor.

This step performs three actions:
1. Gets the Application Insights connection string from the project
2. Configures Azure Monitor OpenTelemetry (with fallback to trace-only exporter)
3. Instruments `azure-ai-projects` and OpenAI-compatible client calls

### Rerun-safe notes
- This section is hardened for repeated execution in the same kernel.
- If full `configure_azure_monitor()` is slow or fails, fallback exporter setup keeps tracing active.
- If telemetry behaves unexpectedly after many reruns, restart the kernel and rerun sections in order.

In [ ]:
import os
import time
import logging
import threading
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter

# Must be set BEFORE importing/calling monitor + instrument()
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

# Enable end-to-end correlation between client and service spans
os.environ["AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION"] = "true"
os.environ["AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE"] = "false"

# Disable Azure host detectors that can stall on local machines
os.environ["OTEL_EXPERIMENTAL_RESOURCE_DETECTORS"] = "otel"
os.environ["OTEL_RESOURCE_DETECTORS"] = "otel"

# Optional: useful logs while troubleshooting monitor configuration
logging.getLogger("azure").setLevel(logging.INFO)

# Import after env vars are set
from azure.monitor.opentelemetry import configure_azure_monitor

def _remove_perf_counter_processor_if_present():
    """Best effort cleanup for reruns in the same kernel."""
    try:
        provider = trace.get_tracer_provider()
        active_processor = getattr(provider, "_active_span_processor", None)
        processors = list(getattr(active_processor, "_span_processors", ()))
        if not processors:
            return
        filtered = [
            p for p in processors
            if p.__class__.__name__ != "_PerformanceCountersSpanProcessor"
        ]
        if len(filtered) != len(processors):
            active_processor._span_processors = tuple(filtered)
            print("       Removed stale performance-counter span processor from current kernel")
    except Exception:
        # Never block the notebook on cleanup logic
        pass

_remove_perf_counter_processor_if_present()

# ── Step 1: Get the App Insights connection string ──
t0 = time.time()
print("[1/3] Retrieving Application Insights connection string...")
ai_conn_str = project_client.telemetry.get_application_insights_connection_string()
print(f"       Done ({time.time() - t0:.1f}s)")

# ── Step 2: Configure Azure Monitor (full OTel distro; guarded) ──
t0 = time.time()
print("[2/3] Configuring Azure Monitor OpenTelemetry...")

monitor_state = {"done": False, "error": None}

def _configure_monitor_worker():
    try:
        configure_azure_monitor(
            connection_string=ai_conn_str,
            disable_offline_storage=True,
            enable_live_metrics=False,
            enable_performance_counters=False,
        )
    except Exception as ex:
        monitor_state["error"] = ex
    finally:
        monitor_state["done"] = True

worker = threading.Thread(target=_configure_monitor_worker, daemon=True)
worker.start()
worker.join(timeout=25)

monitor_configured = False
if worker.is_alive():
    print("       Timeout while configuring full Azure Monitor distro. Falling back to trace-only exporter.")
elif monitor_state["error"] is not None:
    ex = monitor_state["error"]
    print(f"       Full distro setup failed ({type(ex).__name__}: {ex}). Falling back to trace-only exporter.")
else:
    monitor_configured = True
    print(f"       Done ({time.time() - t0:.1f}s)")

if not monitor_configured and not globals().get("_ai_fallback_exporter_configured", False):
    fallback_t0 = time.time()
    exporter = AzureMonitorTraceExporter(connection_string=ai_conn_str)
    provider = trace.get_tracer_provider()
    if hasattr(provider, "add_span_processor"):
        provider.add_span_processor(BatchSpanProcessor(exporter))
    else:
        fallback_provider = TracerProvider()
        fallback_provider.add_span_processor(BatchSpanProcessor(exporter))
        trace.set_tracer_provider(fallback_provider)
    globals()["_ai_fallback_exporter_configured"] = True
    print(f"       Fallback trace exporter ready ({time.time() - fallback_t0:.1f}s)")

# ── Step 3: Instrument the SDK ──
t0 = time.time()
print("[3/3] Instrumenting azure-ai-projects + OpenAI SDK...")
try:
    AIProjectInstrumentor().uninstrument()
except Exception:
    pass

AIProjectInstrumentor().instrument(
    enable_content_recording=True,
    enable_trace_context_propagation=True,
    enable_baggage_propagation=False,
)
print(f"       Done ({time.time() - t0:.1f}s)")

tracer = trace.get_tracer(__name__)
print("Tracer ready ✅")

## 4. Create the Agent

Set `agent_name` and `model_name` in this step, then create (or version-bump) the agent.

- `create_version` creates a new agent or bumps the version if inputs changed.
- Agent behavior is controlled by the `instructions` text.

In [ ]:
agent_name = "ZoDEfendersAgent-1702"
model_name = "gpt-4.1-mini"
agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context."
)

# Wrap agent creation in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-creation") as span:
    span.set_attribute("agent.name", agent_name)
    span.set_attribute("gen_ai.request.model", model_name)

    # Creates an agent, bumps the agent version if parameters have changed
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=agent_instructions,
        ),
    )
    span.set_attribute("agent.version", agent.version)
    span.set_attribute("agent.id", agent.id)

print(f"Agent created: {agent.name} (id: {agent.id}, version: {agent.version})")

## 5. Query the Agent

Use the project’s OpenAI-compatible client to run two independent prompts against the same agent.

- The agent is referenced by name using `agent_reference`.
- Pass 1 generates a fictional story only.
- Pass 2 retrieves factual MSFT Learn insights only.
- Each run is appended to `stories.json` with `story`, `msft_learn_insights`, and `combined_output`.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

openai_client = project_client.get_openai_client()

story_prompt = (
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, "
    "Germany who moonlights as a cybernetic superhero called 'Die DEfender', protector of "
    "the digital realm. Include a plot twist in the last sentence. Always refer to the two "
    "characters by their full names."
)

def parse_response(response) -> tuple[str, list[str], list[str], object]:
    output_items = getattr(response, "output", None) or []
    output_types = [getattr(item, "type", type(item).__name__) for item in output_items]

    approval_ids = [
        item.id
        for item in output_items
        if getattr(item, "type", None) == "mcp_approval_request" and getattr(item, "id", None)
    ]

    text_parts = []
    for item in output_items:
        for content in getattr(item, "content", None) or []:
            text_obj = getattr(content, "text", None)
            if isinstance(text_obj, str) and text_obj.strip():
                text_parts.append(text_obj)
            elif hasattr(text_obj, "value") and isinstance(text_obj.value, str) and text_obj.value.strip():
                text_parts.append(text_obj.value)

    text = (getattr(response, "output_text", None) or "\n".join(text_parts)).strip()
    return text, approval_ids, output_types, getattr(response, "incomplete_details", None)

def query_agent_with_auto_mcp_approval(openai_client, agent_reference_payload, prompt, max_rounds=5):
    response = openai_client.responses.create(
        input=[{"role": "user", "content": prompt}],
        extra_body=agent_reference_payload,
    )

    for round_number in range(1, max_rounds + 1):
        text, approval_ids, output_types, incomplete_details = parse_response(response)
        status = getattr(response, "status", "unknown")

        print(f"Response status: {status}")
        print(f"Response output types: {output_types}")
        if incomplete_details:
            print(f"Incomplete details: {incomplete_details}")

        if text or not approval_ids:
            return response, text

        print(f"Approving MCP tool requests (round {round_number}): {approval_ids}")
        response = openai_client.responses.create(
            previous_response_id=response.id,
            input=[
                {
                    "type": "mcp_approval_response",
                    "approve": True,
                    "approval_request_id": request_id,
                }
                for request_id in approval_ids
            ],
            extra_body=agent_reference_payload,
        )

    final_text, _, _, _ = parse_response(response)
    print("⚠️ Reached maximum MCP approval rounds without assistant text output.")
    return response, final_text

# Wrap the agent query in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-query") as span:
    span.set_attribute("agent.name", agent.name)
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.prompt", story_prompt)

    agent_reference_payload = {
        "agent_reference": {
            "name": agent.name,
            "id": agent.id,
            "type": "agent_reference",
        }
    }

    response, assistant_text = query_agent_with_auto_mcp_approval(
        openai_client=openai_client,
        agent_reference_payload=agent_reference_payload,
        prompt=story_prompt,
        max_rounds=5,
    )

    span.set_attribute("gen_ai.completion", assistant_text[:500] if assistant_text else "")
    span.set_attribute("gen_ai.response.id", response.id)

if not assistant_text:
    print("⚠️ No assistant text was returned.")

print(f"Response output: {assistant_text}")

def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
story_id = append_story(
    stories_file,
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": agent.name,
        "model": model_name,
        "prompt": story_prompt,
        "story": assistant_text,
    },
)

print(f"\nStory #{story_id} saved to {stories_file}")